# 🧪 Optimisation Fine : Poisson Supremacy (Mode Dev)

**Objectif** : Maximiser le F1-score via une recherche exhaustive de la meilleure configuration (Variables + Hyperparamètres + Seuil) pour le modèle agrégé XGBoost Tweedie.

**Contraintes** :
- Pas de fuite (Aggregation *post-split*).
- Évaluation sur données brutes.
- Pas de règles manuelles.

---

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
import itertools
import warnings

warnings.filterwarnings('ignore')
SEED = 42
N_FOLDS = 5

# Chargement des données brutes
df_train = pd.read_csv('conversion_data_train.csv')
df_test = pd.read_csv('conversion_data_test.csv')

# Preprocessing de Base (Identique pour tous)
for df in [df_train, df_test]:
    df['age_bin'] = pd.cut(df['age'], bins=[0, 18, 25, 30, 35, 40, 45, 50, 60, 100], labels=False).fillna(-1).astype(int)
    df['pages_bin'] = pd.cut(df['total_pages_visited'], bins=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16, 18, 20, 25, 100], labels=False).fillna(-1).astype(int)

# Encodage
cat_cols = ['country', 'source']
for col in cat_cols:
    le = LabelEncoder()
    full = pd.concat([df_train[col], df_test[col]])
    le.fit(full)
    df_train[col] = le.transform(df_train[col])
    df_test[col] = le.transform(df_test[col])

print("Données chargées et pré-traitées.")

Données chargées et pré-traitées.


## 📌 Cellule 1 — Modèle de référence

**Configuration Base** :
- Features: `['country', 'source', 'new_user', 'age_bin', 'pages_bin']`
- Model: XGBoost Regressor (Tweedie 1.5)
- Threshold: 0.38 (Empirique)

In [2]:
def train_evaluate_agg(config, X_raw, y_raw):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    f1_scores = []
    auc_scores = []
    
    features = config['features']
    params = config['params']
    threshold_range = config.get('threshold_range', [0.38])
    
    # Aggregation Params
    agg_cols = features # Grouper par les features utilisées
    
    # Note: On a besoin d'optimiser le seuil, donc on va stocker les preds OOF
    oof_preds = np.zeros(len(X_raw))
    oof_targets = np.zeros(len(X_raw))
    
    idx_global = 0
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X_raw, y_raw)):
        # 1. Split Raw
        X_tr_raw, y_tr_raw = X_raw.iloc[train_idx], y_raw.iloc[train_idx]
        X_val_raw, y_val_raw = X_raw.iloc[val_idx], y_raw.iloc[val_idx]

        # 2. Add Target needed for aggregation
        train_df = X_tr_raw.copy()
        train_df['converted'] = y_tr_raw

        # 3. Aggregate TRAIN only (Target Leakage Prevention!)
        # On groupe par les features pour créer les profils
        agg_train = train_df.groupby(agg_cols)['converted'].agg(['mean', 'count']).reset_index()
        agg_train.rename(columns={'mean': 'conversion_rate', 'count': 'weight'}, inplace=True)
        
        # 4. Train Model on Aggregated Data
        model = xgb.XGBRegressor(
            objective='reg:tweedie',
            tweedie_variance_power=1.5,
            **params,
            n_jobs=-1,
            random_state=SEED
        )
        
        model.fit(
            agg_train[agg_cols],
            agg_train['conversion_rate'],
            sample_weight=agg_train['weight']
        )
        
        # 5. Predict on RAW Validation (Join is implicit via predict request)
        # Le modèle prédit sur les features du profil de chaque ligne
        val_preds = model.predict(X_val_raw[agg_cols])
        
        # Store for global threshold opt (approximation, better is fold-wise but costly)
        # Pour l'évaluation simple ici, on enregistre les scores
        auc = roc_auc_score(y_val_raw, val_preds)
        auc_scores.append(auc)
        
        oof_preds[val_idx] = val_preds

    # Global Threshold Optimization
    best_f1 = 0
    best_th = 0
    for th in threshold_range:
        preds_bin = (oof_preds >= th).astype(int)
        score = f1_score(y_raw, preds_bin)
        if score > best_f1:
            best_f1 = score
            best_th = th
            
    return best_f1, np.mean(auc_scores), best_th

# Reference Config
base_features = ['country', 'source', 'new_user', 'age_bin', 'pages_bin']
ref_config = {
    'features': base_features,
    'params': {'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 600}, # Moyen standard
    'threshold_range': np.linspace(0.30, 0.45, 30)
}

print("Evaluation Modèle de Référence...")
ref_f1, ref_auc, ref_th = train_evaluate_agg(ref_config, df_train.drop(columns='converted'), df_train['converted'])
print(f"📌 [REF] F1: {ref_f1:.5f} | AUC: {ref_auc:.5f} | Best TH: {ref_th:.4f}")

Evaluation Modèle de Référence...
📌 [REF] F1: 0.76284 | AUC: 0.98469 | Best TH: 0.3724


## 📌 Cellule 2 — Variables / Configurations à tester

Définition de l'espace de recherche.

In [3]:
# 1. Feature Sets
# V1 = Base
# V2 = Base (Mais on force l'ajout de variables dérivées dans le DF avant)

# Préparation des Features Dérivées pour V2/V3
df_train['age_pages_inter'] = df_train['age_bin'] * df_train['pages_bin']
df_train['pages_age_ratio'] = df_train['total_pages_visited'] / (df_train['age'] + 1)

# List of Feature Configurations
feature_sets = {
    'V1 (Base)': base_features,
    'V2 (Inter)': base_features + ['age_pages_inter'],
    'V3 (Ratio)': base_features + ['pages_age_ratio']
}

# 2. Hyperparameters
param_grid = {
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [4, 5, 6],
    'n_estimators': [600, 1000]
}

# 3. Thresholds
fine_thresholds = np.arange(0.30, 0.42, 0.002)

# Generate all Configs
configs = []
keys, values = zip(*param_grid.items())
permutations = [dict(zip(keys, v)) for v in itertools.product(*values)]

for feat_name, feats in feature_sets.items():
    for p in permutations:
        c = {
            'name': f"{feat_name} | lr={p['learning_rate']} d={p['max_depth']} n={p['n_estimators']}",
            'features': feats,
            'params': p,
            'threshold_range': fine_thresholds
        }
        configs.append(c)

print(f"🚀 Nombre total de configurations à tester : {len(configs)}")

🚀 Nombre total de configurations à tester : 54


## 📌 Cellule 3 — Évaluation automatique

Boucle d'optimisation.

In [4]:
results = []

print("Démarrage de l'optimisation (patience requise)...")

for i, conf in enumerate(configs):
    # Run CV
    # Note: On doit passer les colonnes correctes
    # df_train contient toutes les nouvelles colonnes
    f1, auc, th = train_evaluate_agg(
        conf, 
        df_train.drop(columns='converted'), 
        df_train['converted']
    )
    
    res = {
        'Config': conf['name'],
        'F1 Only': f1,
        'AUC': auc,
        'Best TH': th,
        'Params': conf['params'],
        'Features': conf['features']
    }
    results.append(res)
    
    print(f"[{i+1}/{len(configs)}] {conf['name'][:40]}... -> F1: {f1:.5f} (TH: {th:.3f})")

df_res = pd.DataFrame(results).sort_values(by='F1 Only', ascending=False)

print("\n🏁 Optimisation Terminée.")

Démarrage de l'optimisation (patience requise)...
[1/54] V1 (Base) | lr=0.03 d=4 n=600... -> F1: 0.76320 (TH: 0.352)
[2/54] V1 (Base) | lr=0.03 d=4 n=1000... -> F1: 0.76276 (TH: 0.354)
[3/54] V1 (Base) | lr=0.03 d=5 n=600... -> F1: 0.76320 (TH: 0.346)
[4/54] V1 (Base) | lr=0.03 d=5 n=1000... -> F1: 0.76281 (TH: 0.356)
[5/54] V1 (Base) | lr=0.03 d=6 n=600... -> F1: 0.76271 (TH: 0.356)
[6/54] V1 (Base) | lr=0.03 d=6 n=1000... -> F1: 0.76236 (TH: 0.400)
[7/54] V1 (Base) | lr=0.05 d=4 n=600... -> F1: 0.76258 (TH: 0.350)
[8/54] V1 (Base) | lr=0.05 d=4 n=1000... -> F1: 0.76302 (TH: 0.354)
[9/54] V1 (Base) | lr=0.05 d=5 n=600... -> F1: 0.76289 (TH: 0.376)
[10/54] V1 (Base) | lr=0.05 d=5 n=1000... -> F1: 0.76246 (TH: 0.382)
[11/54] V1 (Base) | lr=0.05 d=6 n=600... -> F1: 0.76261 (TH: 0.398)
[12/54] V1 (Base) | lr=0.05 d=6 n=1000... -> F1: 0.76222 (TH: 0.392)
[13/54] V1 (Base) | lr=0.1 d=4 n=600... -> F1: 0.76292 (TH: 0.356)
[14/54] V1 (Base) | lr=0.1 d=4 n=1000... -> F1: 0.76185 (TH: 0.360)
[1

## 📌 Cellule 4 — Sélection finale

Analyse et Choix du Vainqueur.

In [5]:
print("🏆 TOP 5 CONFIGURATIONS :")
print(df_res[['Config', 'F1 Only', 'AUC', 'Best TH']].head(5))

best_config = df_res.iloc[0]

print("\n" + "="*40)
print("🥇 CONFIGURATION GAGNANTE")
print("="*40)
print(f"Description : {best_config['Config']}")
print(f"F1 Score    : {best_config['F1 Only']:.5f}")
print(f"AUC Score   : {best_config['AUC']:.5f}")
print(f"Seuil Opt   : {best_config['Best TH']:.4f}")

print("\nComparaison vs Référence :")
print(f"Gain F1 : {best_config['F1 Only'] - ref_f1:+.5f}")

🏆 TOP 5 CONFIGURATIONS :
                             Config   F1 Only       AUC  Best TH
40   V3 (Ratio) | lr=0.03 d=6 n=600  0.766573  0.984526    0.384
39  V3 (Ratio) | lr=0.03 d=5 n=1000  0.766283  0.984536    0.384
38   V3 (Ratio) | lr=0.03 d=5 n=600  0.766243  0.985097    0.380
44   V3 (Ratio) | lr=0.05 d=5 n=600  0.766239  0.984604    0.380
46   V3 (Ratio) | lr=0.05 d=6 n=600  0.765904  0.983877    0.404

🥇 CONFIGURATION GAGNANTE
Description : V3 (Ratio) | lr=0.03 d=6 n=600
F1 Score    : 0.76657
AUC Score   : 0.98453
Seuil Opt   : 0.3840

Comparaison vs Référence :
Gain F1 : +0.00373
